# Etapa 3 · Preparação dos dados (Extended Essay-BR)

**Objetivo:** transformar o corpus em três CSVs limpos (`data/train.csv`, `data/dev.csv`, `data/test.csv`) num formato único, lido por todos os modelos.

**Fonte:** [lplnufpi/essay-br](https://github.com/lplnufpi/essay-br), versão estendida (Marinho, Anchiêta e Moura, *JIDM*, 2022), fixada no commit `7f0ea9c`.

Toda a lógica está em `src/dados.py`; este notebook executa cada passo e mostra os números que vão para o relatório.

> A partir do fim deste notebook, **o teste fica congelado**: nenhum ajuste de modelo é feito olhando para ele.

In [1]:
# Configuração: funciona localmente (raiz do repositório) e no Colab
import sys, os
from pathlib import Path

if "google.colab" in sys.modules:
    !git clone -q https://github.com/thmlazaro/tp2-ia-2026.2.git /content/repo
    RAIZ = Path("/content/repo")
else:
    RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import dados
from utils import COMPETENCIAS as C, NOTAS_VALIDAS
pd.set_option("display.width", 140)
print("raiz do projeto:", RAIZ.name, "| dados em", (RAIZ / "data").relative_to(RAIZ))

raiz do projeto: tp2-ia-2026.2 | dados em data


## 3.1 Carregar o corpus e gerar os splits oficiais

O repositório dos autores não traz os splits prontos: eles são gerados pelo script `build_dataset.py`, que aplica `train_test_split` duas vezes com `random_state=230` (70% treino, 15% dev, 15% teste). `dados.gerar_splits_oficiais` reproduz exatamente esse procedimento.

In [2]:
corpus, prompts = dados.baixar_corpus(RAIZ / "data" / "raw")
print("redações no corpus:", len(corpus), "| temas:", len(prompts))
print("colunas:", corpus.columns.tolist())

brutos = dados.gerar_splits_oficiais(corpus)
for nome, d in zip(["treino", "dev", "teste"], brutos):
    print(f"{nome:6s}: {len(d)}")

# os splits são determinísticos: gerar de novo dá exatamente as mesmas redações
de_novo = dados.gerar_splits_oficiais(corpus)
print("determinístico:", all(a.index.equals(b.index) for a, b in zip(brutos, de_novo)))

redações no corpus: 6577 | temas: 151
colunas: ['prompt', 'title', 'essay', 'c1', 'c2', 'c3', 'c4', 'c5', 'score']
treino: 4603
dev   : 987
teste : 987
determinístico: True


## 3.2 Formato padrão

- parágrafos unidos num único texto, separados por `\n` (e espaços não separáveis `\xa0` normalizados);
- notas em `c1`…`c5` e total em `score`;
- `id` estável = posição da redação no arquivo original (`essay_00697` é sempre a mesma redação);
- `prompt` (id do tema) e `tema` (título da proposta), usados pelo MariTalk e na análise de limitações.

In [3]:
train, dev, test = (dados.preparar(d, prompts) for d in brutos)
train.head(3)

,id,texto,c1,c2,c3,c4,c5,score,prompt,tema
0,essay_05040,É notório que as redes sociais estão cada vez ...,120,120,120,160,160,680,86,Impactos do uso das redes sociais na saúde mental
1,essay_04841,Com o avanço das tecnologias ao redor do mundo...,200,160,160,200,160,880,86,Impactos do uso das redes sociais na saúde mental
2,essay_04031,A declaração universal dos direitos humanos af...,160,160,160,160,160,800,47,Desafios na Educação a Distância no Brasil


## 3.3 Verificações de qualidade

In [4]:
pd.DataFrame({n: dados.verificar(d) for n, d in [("treino", train), ("dev", dev), ("teste", test)]}).T

,n,notas_invalidas,soma_diferente_score,vazias,menos_de_50_palavras,palavras_media
treino,4603.0,0.0,0.0,8.0,8.0,289.291549
dev,987.0,0.0,0.0,0.0,0.0,290.972644
teste,987.0,0.0,0.0,2.0,2.0,287.210740


Nenhuma nota fora de {0, 40, …, 200} e nenhuma soma divergente do total. Os únicos problemas são **textos vazios**: a redação não existe no corpus, embora tenha nota. Como não há entrada para os modelos, essas redações são removidas (também do teste, pois não podem ser avaliadas). Todas as demais têm mais de 50 palavras.

In [5]:
vazias = pd.concat([d[d.texto.str.strip() == ""].assign(split=n)
                    for n, d in [("treino", train), ("dev", dev), ("teste", test)]])
vazias[["split", "id", "score"] + C]

,split,id,score,c1,c2,c3,c4,c5
1010,treino,essay_06278,360,120,120,120,0,0
1193,treino,essay_06228,720,160,160,120,120,160
1902,treino,essay_06231,400,120,120,80,80,0
2093,treino,essay_06409,680,160,160,120,120,120
2664,treino,essay_06478,360,80,120,80,80,0
3293,treino,essay_05853,360,80,120,80,80,0
3634,treino,essay_05995,520,120,120,80,80,120
4263,treino,essay_06404,680,160,160,120,120,120
424,teste,essay_06411,880,200,160,160,200,160
791,teste,essay_05767,640,120,120,160,160,80


In [6]:
train, dev, test = dados.limpar(train), dados.limpar(dev), dados.limpar(test)
print({n: len(d) for n, d in [("treino", train), ("dev", dev), ("teste", test)]})

{'treino': 4595, 'dev': 987, 'teste': 985}


### Duplicatas entre os splits

Textos idênticos (após normalizar caixa e espaços) em splits diferentes inflariam as métricas. **Regra:** a duplicata sai do treino (ou do dev), nunca do teste.

In [7]:
(train, dev, test), rel_dup = dados.remover_duplicatas(train, dev, test)
rel_dup

{'treino_e_teste': 0,
 'treino_e_dev': 1,
 'dev_e_teste': 0,
 'internas_treino': 2,
 'internas_dev': 0,
 'internas_teste': 0,
 'removidas': {'train': 3, 'dev': 0, 'test': 0}}

### Temas repetidos

Temas repetidos não são duplicatas (são redações diferentes sobre a mesma proposta), então não são removidos. Mas vão para as limitações: o modelo é avaliado em temas que já viu no treino.

In [8]:
frac = test["prompt"].isin(set(train["prompt"])).mean()
print(f"temas no treino: {train.prompt.nunique()} | no teste: {test.prompt.nunique()}")
print(f"{frac:.1%} das redações de teste têm um tema que também aparece no treino")

temas no treino: 151 | no teste: 143
100.0% das redações de teste têm um tema que também aparece no treino


## 3.4 Tamanho em tokens no BERTimbau

O BERT lê no máximo 512 tokens. A fração de redações acima disso será truncada no fine-tuning (limitação documentada).

In [9]:
import warnings; warnings.filterwarnings("ignore")   # avisos de bibliotecas, irrelevantes aqui
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()   # só contamos tokens; o aviso de sequência longa não se aplica
tok = AutoTokenizer.from_pretrained(dados.MODELO_TOKENIZER)

linhas = {}
for nome, d in [("treino", train), ("dev", dev), ("teste", test)]:
    d["n_tokens"] = dados.contar_tokens(d, tok)
    n = d["n_tokens"]
    linhas[nome] = {"média": round(n.mean()), "mediana": n.median(), "p90": n.quantile(.9),
                    "máx": n.max(), "> 512 tokens": f"{(n > 512).mean():.1%}"}
pd.DataFrame(linhas).T

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


,média,mediana,p90,máx,> 512 tokens
treino,378,372.0,522.0,1143,11.7%
dev,379,370.0,535.0,926,13.9%
teste,375,371.0,519.0,882,11.3%


## 3.5 Distribuição das notas (figura do relatório)

In [10]:
%matplotlib inline
fig = dados.grafico_distribuicao(train, RAIZ / "results" / "figuras" / "dist_notas.png")

In [11]:
# frequência relativa (%) de cada nota no treino: mostra o desbalanceamento
(train[C].apply(lambda s: s.value_counts(normalize=True)).reindex(list(NOTAS_VALIDAS)).fillna(0) * 100).round(1)

,c1,c2,c3,c4,c5
0,1.7,1.9,2.7,3.1,7.4
40,0.3,1.3,2.5,0.9,4.7
80,7.9,14.4,24.7,13.7,20.1
120,47.8,37.3,46.5,37.2,35.1
160,37.7,36.8,20.5,27.7,23.6
200,4.6,8.5,3.1,17.4,9.1


**Leitura:** 120 e 160 concentram a maioria das notas em todas as competências; 0 e 40 são raros (exceto em C5, a proposta de intervenção, onde 0 aparece em 7,4% das redações). Um modelo que sempre "chuta" 120 já consegue um erro médio baixo, por isso a métrica principal será o QWK (etapa 6).

## 3.6 Salvar

Executa o pipeline completo de `src/dados.py` (os mesmos passos acima), que grava os CSVs e `results/estatisticas_dados.json` com todos os números desta etapa.

In [12]:
train, dev, test, est = dados.executar(RAIZ / "data", RAIZ / "results")
print("final:", est["final"])
for nome in ["train", "dev", "test"]:
    d = pd.read_csv(RAIZ / "data" / f"{nome}.csv")
    assert d[C].isin(NOTAS_VALIDAS).all().all() and (d[C].sum(axis=1) == d.score).all()
    assert d.texto.str.strip().str.len().min() > 0 and d.notna().all().all() and d.id.is_unique
print("CSVs relidos e validados:", list(d.columns))

final: {'train': 4592, 'dev': 987, 'test': 985}
CSVs relidos e validados: ['id', 'texto', 'c1', 'c2', 'c3', 'c4', 'c5', 'score', 'prompt', 'tema', 'n_tokens']
